In [ ]:
%%capture
import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
%%capture
!pip uninstall mamba-ssm causal-conv1d torch torchvision torchaudio -y
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
!pip cache purge
MAX_JOBS=1
!pip install causal-conv1d>=1.4.0 --no-cache-dir --no-build-isolation --no-binary :all:
MAX_JOBS=1
!pip install mamba-ssm --no-cache-dir --no-build-isolation --no-binary :all:

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import re
import json
import math
import random
import pickle
import shutil
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict
from datetime import datetime
from collections import Counter
from copy import deepcopy

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import torchvision.transforms as transforms

from mamba_ssm import Mamba

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_gpus = torch.cuda.device_count()

print(f'Device: {device}')
print(f'Available GPUs: {num_gpus}')
if torch.cuda.is_available():
    for i in range(num_gpus):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} | {props.total_memory // 1024**3} GB VRAM')

In [ ]:
DATA_ROOT = copy_to_fast_storage(
    '/kaggle/input/datasets/sambhavnayak/genesys-spectrogram-dataset',
    '/dev/shm/genesys_data'
)

In [ ]:
def copy_to_fast_storage(src, dest):
    src_path = Path(src)
    dest_path = Path(dest)
    if dest_path.exists():
        print(f'Data already in {dest_path}')
        return str(dest_path)
    print(f'Copying {src_path} to {dest_path}... (This takes ~2-5 mins but saves hours)')
    os.system(f"cp -r '{src_path}' '{dest_path}'")
    print('Copy complete.')
    return str(dest_path)

DATA_ROOT = copy_to_fast_storage(
    '/kaggle/input/datasets/sambhavnayak/genesys-spectrogram-dataset',
    '/dev/shm/genesys_data'
)

UAV_NAMES      = [f'uav{i}' for i in range(1, 8)]
NUM_UAV        = len(UAV_NAMES)
UAV_TO_IDX     = {n: i for i, n in enumerate(UAV_NAMES)}
IDX_TO_UAV     = {i: n for n, i in UAV_TO_IDX.items()}

DIST_NAMES     = ['6ft', '9ft', '12ft', '15ft']
NUM_DIST       = len(DIST_NAMES)
DIST_TO_IDX    = {d: i for i, d in enumerate(DIST_NAMES)}

SNR_NAMES      = ['Clean', '0', '5', '10', '15', '20']
NUM_SNR        = len(SNR_NAMES)
SNR_TO_IDX     = {s: i for i, s in enumerate(SNR_NAMES)}
IDX_TO_SNR     = {i: s for s, i in SNR_TO_IDX.items()}

print('UAV classes:')
for i, n in IDX_TO_UAV.items():
    print(f'  {i}: {n}')
print(f'Distance classes : {DIST_NAMES}')
print(f'SNR classes      : {SNR_NAMES}')

@dataclass
class Config:
    img_size:              int   = 224
    patch_size:            int   = 4
    in_channels:           int   = 3
    num_device_classes:    int   = NUM_UAV
    num_dist_classes:      int   = NUM_DIST
    num_snr_classes:       int   = NUM_SNR
    embed_dim:             int   = 96
    depth:                 int   = 4
    d_state:               int   = 16
    d_conv:                int   = 4
    expand:                int   = 2
    drop_rate:             float = 0.3
    drop_path_rate:        float = 0.1
    layer_scale_init:      float = 0.1
    batch_size:            int   = 32
    learning_rate:         float = 2e-4
    weight_decay:          float = 0.001
    epochs:                int   = 80
    patience:              int   = 25
    grad_clip_norm:        float = 1.0
    label_smoothing:       float = 0.1
    warmup_epochs:         int   = 15
    val_split:             float = 0.20
    random_state:          int   = 42
    loss_w_device:         float = 1.0
    loss_w_dist:           float = 0.5
    loss_w_snr:            float = 0.3

config = Config()

print(f'\nConfig loaded')
print(f'  embed_dim={config.embed_dim}  depth={config.depth}  d_state={config.d_state}')
print(f'  lr={config.learning_rate}  epochs={config.epochs}  batch_size={config.batch_size}')
print(f'  patience={config.patience}  warmup={config.warmup_epochs}')
print(f'  loss weights -> device={config.loss_w_device}  dist={config.loss_w_dist}  snr={config.loss_w_snr}')

In [ ]:
def parse_filename(filename: str) -> Tuple[str, str]:
    stem = Path(filename).stem

    dist_match = re.search(r'(6ft|9ft|12ft|15ft)', stem)
    dist_str = dist_match.group(1) if dist_match else DIST_NAMES[0]

    snr_match = re.search(r'snr(Clean|\d+)', stem)
    if snr_match:
        snr_str = snr_match.group(1)
    else:
        snr_str = 'Clean'

    return dist_str, snr_str


def load_dataset(data_root: str) -> Tuple[List[str], np.ndarray, np.ndarray, np.ndarray]:
    paths, uav_labels, dist_labels, snr_labels = [], [], [], []
    print(f'\nLoading dataset from: {data_root}')
    for uav_name in UAV_NAMES:
        uav_dir = Path(data_root) / uav_name
        if not uav_dir.exists():
            print(f'  WARNING: Missing folder: {uav_dir}')
            continue
        png_files = sorted(uav_dir.glob('*.png'))
        uav_idx = UAV_TO_IDX[uav_name]
        for p in png_files:
            dist_str, snr_str = parse_filename(p.name)
            dist_idx = DIST_TO_IDX.get(dist_str, 0)
            snr_idx  = SNR_TO_IDX.get(snr_str, 0)
            paths.append(str(p))
            uav_labels.append(uav_idx)
            dist_labels.append(dist_idx)
            snr_labels.append(snr_idx)
        print(f'  Loading {uav_name}... {len(png_files)} images')

    uav_labels  = np.array(uav_labels,  dtype=np.int64)
    dist_labels = np.array(dist_labels, dtype=np.int64)
    snr_labels  = np.array(snr_labels,  dtype=np.int64)
    print(f'Total images loaded: {len(paths)}')
    print(f'UAV dist  : {dict(Counter(dist_labels.tolist()))}')
    print(f'SNR dist  : {dict(Counter(snr_labels.tolist()))}')
    return paths, uav_labels, dist_labels, snr_labels


all_paths, all_uav, all_dist, all_snr = load_dataset(DATA_ROOT)

In [ ]:
indices = np.arange(len(all_paths))
train_idx, val_idx = train_test_split(
    indices,
    test_size=config.val_split,
    random_state=config.random_state,
    stratify=all_uav,
)
train_idx = sorted(train_idx.tolist())
val_idx   = sorted(val_idx.tolist())

train_paths  = [all_paths[i] for i in train_idx]
train_uav    = all_uav[train_idx]
train_dist   = all_dist[train_idx]
train_snr    = all_snr[train_idx]

val_paths    = [all_paths[i] for i in val_idx]
val_uav      = all_uav[val_idx]
val_dist     = all_dist[val_idx]
val_snr      = all_snr[val_idx]

ALL_SNR_LEVELS = sorted(set(all_snr.tolist()))

print(f'Train: {len(train_paths)} | Val (held-out 20%): {len(val_paths)}')
print(f'Train UAV counts : {dict(Counter(train_uav.tolist()))}')
print(f'Val   UAV counts : {dict(Counter(val_uav.tolist()))}')

In [ ]:
def compute_spectrogram_stats(paths: List[str],
                               sample_size: int = 2000) -> Tuple[List[float], List[float]]:
    rng  = np.random.default_rng(42)
    idx  = rng.choice(len(paths), min(sample_size, len(paths)), replace=False)
    to_t = transforms.ToTensor()

    pixel_lists = [[] for _ in range(3)]

    for i in idx:
        img = to_t(Image.open(paths[i]).convert('RGB'))
        for c in range(3):
            pixel_lists[c].append(img[c].reshape(-1))

    mean, std = [], []
    for c in range(3):
        all_pixels = torch.cat(pixel_lists[c])
        mean.append(all_pixels.mean().item())
        std.append(all_pixels.std().item())

    return mean, std


print('Computing spectrogram normalisation statistics from training set...')
SPEC_MEAN, SPEC_STD = compute_spectrogram_stats(train_paths, sample_size=2000)
print(f'  SPEC_MEAN = {[round(v, 4) for v in SPEC_MEAN]}')
print(f'  SPEC_STD  = {[round(v, 4) for v in SPEC_STD]}')

In [ ]:
class SpecAugment:
    def __init__(self,
                 freq_mask_param: int = 20,
                 time_mask_param: int = 20,
                 num_freq_masks:  int = 2,
                 num_time_masks:  int = 2):
        self.F  = freq_mask_param
        self.T  = time_mask_param
        self.nF = num_freq_masks
        self.nT = num_time_masks

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        _, H, W = x.shape
        for _ in range(self.nF):
            f  = random.randint(0, self.F)
            f0 = random.randint(0, max(H - f, 0))
            x[:, f0:f0 + f, :] = 0.0
        for _ in range(self.nT):
            t  = random.randint(0, self.T)
            t0 = random.randint(0, max(W - t, 0))
            x[:, :, t0:t0 + t] = 0.0
        return x


def get_train_transforms(img_size: int = 224) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=SPEC_MEAN, std=SPEC_STD),
        SpecAugment(freq_mask_param=20,
                    time_mask_param=20,
                    num_freq_masks=2,
                    num_time_masks=2),
    ])


def get_val_transforms(img_size: int = 224) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=SPEC_MEAN, std=SPEC_STD),
    ])


class GenesysDataset(Dataset):
    def __init__(self, paths: List[str],
                 uav_labels: np.ndarray,
                 dist_labels: np.ndarray,
                 snr_labels: np.ndarray,
                 transform=None):
        self.paths       = paths
        self.uav_labels  = torch.tensor(uav_labels,  dtype=torch.long)
        self.dist_labels = torch.tensor(dist_labels, dtype=torch.long)
        self.snr_labels  = torch.tensor(snr_labels,  dtype=torch.long)
        self.transform   = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.uav_labels[idx], self.dist_labels[idx], self.snr_labels[idx]


def build_weighted_sampler(labels: np.ndarray) -> WeightedRandomSampler:
    class_counts = Counter(labels.tolist())
    weights = [1.0 / class_counts[int(lb)] for lb in labels]
    return WeightedRandomSampler(weights, len(weights), replacement=True)


def build_dataloaders(cfg: Config,
                      tr_paths, tr_uav, tr_dist, tr_snr,
                      vl_paths, vl_uav, vl_dist, vl_snr,
                      num_workers: int = 4):
    train_ds = GenesysDataset(tr_paths, tr_uav, tr_dist, tr_snr,
                              transform=get_train_transforms(cfg.img_size))
    val_ds   = GenesysDataset(vl_paths, vl_uav, vl_dist, vl_snr,
                              transform=get_val_transforms(cfg.img_size))

    sampler = build_weighted_sampler(tr_uav)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size,
                              sampler=sampler, num_workers=num_workers,
                              pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size,
                              shuffle=False, num_workers=num_workers,
                              pin_memory=True)
    return train_loader, val_loader


_sanity_train_loader, _sanity_val_loader = build_dataloaders(
    config,
    train_paths, train_uav, train_dist, train_snr,
    val_paths,   val_uav,   val_dist,   val_snr,
)

print(f'Train batches : {len(_sanity_train_loader)}')
print(f'Val   batches : {len(_sanity_val_loader)}')

imgs, l_uav, l_dist, l_snr = next(iter(_sanity_train_loader))
print(f'Batch image shape : {imgs.shape}')
print(f'UAV labels sample : {l_uav[:8].tolist()}')
print(f'Dist labels sample: {l_dist[:8].tolist()}')
print(f'SNR labels sample : {l_snr[:8].tolist()}')
print(f'Batch image range : [{imgs.min():.3f}, {imgs.max():.3f}]')

del _sanity_train_loader, _sanity_val_loader

In [ ]:
class StochasticDepth(nn.Module):
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.drop_prob == 0.0:
            return x
        keep_prob = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        rand  = torch.rand(shape, dtype=x.dtype, device=x.device)
        rand  = torch.floor(rand + keep_prob)
        return (x / keep_prob) * rand


class LayerScale(nn.Module):
    def __init__(self, dim: int, init_value: float = 0.1):
        super().__init__()
        self.gamma = nn.Parameter(init_value * torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.gamma


class PatchEmbed(nn.Module):
    def __init__(self, img_size: int, patch_size: int,
                 in_chans: int, embed_dim: int):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
        nn.init.xavier_uniform_(self.proj.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x).flatten(2).transpose(1, 2)


class VimBlock(nn.Module):
    def __init__(self, dim: int, d_state: int, d_conv: int, expand: int,
                 drop_path: float = 0.0, layer_scale_init: float = 0.1):
        super().__init__()
        self.norm        = nn.LayerNorm(dim)
        self.mamba_fwd   = Mamba(d_model=dim, d_state=d_state,
                                 d_conv=d_conv, expand=expand)
        self.mamba_bwd   = Mamba(d_model=dim, d_state=d_state,
                                 d_conv=d_conv, expand=expand)
        self.layer_scale = LayerScale(dim, layer_scale_init)
        self.drop_path   = StochasticDepth(drop_path)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        h = self.norm(x)
        y_fwd = self.mamba_fwd(h)
        y_bwd = self.mamba_bwd(h.flip([1])).flip([1])
        out   = self.layer_scale(y_fwd + y_bwd)
        return residual + self.drop_path(out)


class VisionMamba(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        D = cfg.embed_dim

        self.patch_embed = PatchEmbed(cfg.img_size, cfg.patch_size,
                                      cfg.in_channels, D)
        N = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, D))
        self.pos_embed = nn.Parameter(torch.zeros(1, N + 1, D))
        self.pos_drop  = nn.Dropout(p=cfg.drop_rate)

        dpr = [x.item() for x in
               torch.linspace(0, cfg.drop_path_rate, cfg.depth)]

        self.layers = nn.ModuleList([
            VimBlock(
                dim=D,
                d_state=cfg.d_state,
                d_conv=cfg.d_conv,
                expand=cfg.expand,
                drop_path=dpr[i],
                layer_scale_init=cfg.layer_scale_init,
            )
            for i in range(cfg.depth)
        ])

        self.norm       = nn.LayerNorm(D)
        self.head_device = nn.Sequential(
            nn.Dropout(p=cfg.drop_rate),
            nn.Linear(D, cfg.num_device_classes),
        )
        self.head_dist = nn.Sequential(
            nn.Dropout(p=cfg.drop_rate),
            nn.Linear(D, cfg.num_dist_classes),
        )
        self.head_snr = nn.Sequential(
            nn.Dropout(p=cfg.drop_rate),
            nn.Linear(D, cfg.num_snr_classes),
        )

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        B = x.shape[0]
        x = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x   = self.pos_drop(x + self.pos_embed)
        for layer in self.layers:
            x = layer(x)
        x    = self.norm(x)
        feat = x[:, 0]
        return self.head_device(feat), self.head_dist(feat), self.head_snr(feat)


_test_cfg = Config()
_model    = VisionMamba(_test_cfg).to(device)
_inp      = torch.randn(2, 3, 224, 224, device=device)
_out      = _model(_inp)
print(f'VisionMamba output shapes: device={_out[0].shape}  dist={_out[1].shape}  snr={_out[2].shape}')
total_p = sum(p.numel() for p in _model.parameters())
train_p = sum(p.numel() for p in _model.parameters() if p.requires_grad)
print(f'Parameters: {total_p:,} total | {train_p:,} trainable')
del _model, _inp, _out
torch.cuda.empty_cache()

In [ ]:
def adjust_lr(optimizer: torch.optim.Optimizer,
              epoch: int, cfg: Config) -> float:
    if epoch < cfg.warmup_epochs:
        lr = cfg.learning_rate * (epoch + 1) / cfg.warmup_epochs
    else:
        progress = (epoch - cfg.warmup_epochs) / max(
            cfg.epochs - cfg.warmup_epochs, 1)
        lr = cfg.learning_rate * 0.5 * (1.0 + math.cos(math.pi * progress))
    for pg in optimizer.param_groups:
        pg['lr'] = lr
    return lr


def compute_metrics(y_true: np.ndarray,
                    y_pred: np.ndarray,
                    num_classes: int) -> Dict[str, float]:
    acc = accuracy_score(y_true, y_pred) * 100.0
    f1  = f1_score(y_true, y_pred, average='weighted', zero_division=0) * 100.0
    cm  = confusion_matrix(y_true, y_pred, labels=range(num_classes))

    fpr_list, fnr_list = [], []
    for i in range(num_classes):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        fpr_list.append(fp / (fp + tn) if (fp + tn) > 0 else 0.0)
        fnr_list.append(fn / (fn + tp) if (fn + tp) > 0 else 0.0)

    return {
        'accuracy': round(acc, 2),
        'f1':       round(f1,  2),
        'fpr':      round(np.mean(fpr_list) * 100.0, 2),
        'fnr':      round(np.mean(fnr_list) * 100.0, 2),
    }


print('LR schedule and metrics utilities defined')

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0,
                 label_smoothing: float = 0.1,
                 num_classes: int = NUM_UAV):
        super().__init__()
        self.gamma           = gamma
        self.label_smoothing = label_smoothing
        self.num_classes     = num_classes

    def forward(self, logits: torch.Tensor,
                targets: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            smooth_targets = torch.full_like(
                logits, self.label_smoothing / (self.num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1),
                                    1.0 - self.label_smoothing)

        log_prob = F.log_softmax(logits, dim=-1)
        prob     = log_prob.exp()

        ce = -(smooth_targets * log_prob).sum(dim=-1)
        pt = prob.gather(1, targets.unsqueeze(1)).squeeze(1)
        return (((1.0 - pt) ** self.gamma) * ce).mean()


def train_one_epoch(model: nn.Module,
                    loader: DataLoader,
                    crit_device: nn.Module,
                    crit_dist: nn.Module,
                    crit_snr: nn.Module,
                    optimizer: torch.optim.Optimizer,
                    cfg: Config) -> Tuple[float, float]:
    model.train()
    total_loss    = 0.0
    total_correct = 0
    total_samples = 0
    nan_batches   = 0

    for step, (images, lbl_dev, lbl_dist, lbl_snr) in enumerate(loader):
        images   = images.to(device, non_blocking=True)
        lbl_dev  = lbl_dev.to(device, non_blocking=True)
        lbl_dist = lbl_dist.to(device, non_blocking=True)
        lbl_snr  = lbl_snr.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(dtype=torch.bfloat16):
            logits_dev, logits_dist, logits_snr = model(images)
            loss = (
                cfg.loss_w_device * crit_device(logits_dev,  lbl_dev)  +
                cfg.loss_w_dist   * crit_dist(logits_dist, lbl_dist) +
                cfg.loss_w_snr    * crit_snr(logits_snr,  lbl_snr)
            )

        if torch.isnan(loss) or torch.isinf(loss):
            nan_batches += 1
            optimizer.zero_grad(set_to_none=True)
            if nan_batches <= 3:
                print(f'  WARNING: NaN/Inf loss at step {step}. Skipping batch.')
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=cfg.grad_clip_norm)
        optimizer.step()

        bs             = images.size(0)
        total_loss    += loss.item() * bs
        total_correct += (logits_dev.argmax(1) == lbl_dev).sum().item()
        total_samples += bs

    if nan_batches > 0:
        print(f'  [nan_batches total this epoch: {nan_batches}]')

    if total_samples == 0:
        return float('nan'), 0.0
    return total_loss / total_samples, total_correct / total_samples * 100.0


@torch.no_grad()
def evaluate(model: nn.Module,
             loader: DataLoader,
             crit_device: nn.Module,
             crit_dist: nn.Module,
             crit_snr: nn.Module,
             cfg: Config) -> Tuple[float, Dict]:
    model.eval()
    total_loss    = 0.0
    total_samples = 0
    all_preds     = []
    all_labels    = []

    for images, lbl_dev, lbl_dist, lbl_snr in loader:
        images   = images.to(device, non_blocking=True)
        lbl_dev  = lbl_dev.to(device, non_blocking=True)
        lbl_dist = lbl_dist.to(device, non_blocking=True)
        lbl_snr  = lbl_snr.to(device, non_blocking=True)

        with autocast(dtype=torch.bfloat16):
            logits_dev, logits_dist, logits_snr = model(images)
            loss = (
                cfg.loss_w_device * crit_device(logits_dev,  lbl_dev)  +
                cfg.loss_w_dist   * crit_dist(logits_dist, lbl_dist) +
                cfg.loss_w_snr    * crit_snr(logits_snr,  lbl_snr)
            )

        bs             = images.size(0)
        total_loss    += loss.item() * bs
        total_samples += bs
        all_preds.extend(logits_dev.argmax(1).cpu().tolist())
        all_labels.extend(lbl_dev.cpu().tolist())

    metrics = compute_metrics(
        np.array(all_labels), np.array(all_preds), NUM_UAV)
    return total_loss / total_samples, metrics


def run_training(cfg: Config,
                 tr_paths, tr_uav, tr_dist, tr_snr,
                 vl_paths, vl_uav, vl_dist, vl_snr,
                 tag: str = 'vim',
                 verbose: bool = True) -> Dict:
    train_loader, val_loader = build_dataloaders(
        cfg,
        tr_paths, tr_uav, tr_dist, tr_snr,
        vl_paths, vl_uav, vl_dist, vl_snr)

    model = VisionMamba(cfg)
    if num_gpus > 1:
        model = nn.DataParallel(model)
        if verbose:
            print(f'  Using DataParallel on {num_gpus} GPUs')
    model = model.to(device)

    crit_device = FocalLoss(
        gamma=2.0,
        label_smoothing=cfg.label_smoothing,
        num_classes=cfg.num_device_classes,
    ).to(device)

    crit_dist = FocalLoss(
        gamma=2.0,
        label_smoothing=cfg.label_smoothing,
        num_classes=cfg.num_dist_classes,
    ).to(device)

    crit_snr = FocalLoss(
        gamma=2.0,
        label_smoothing=cfg.label_smoothing,
        num_classes=cfg.num_snr_classes,
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc':  [], 'val_acc':  [],
        'lr': [],
    }
    best_val_acc = 0.0
    best_metrics = None
    best_state   = None
    patience_ctr = 0

    if verbose:
        print(f'\n{"="*70}')
        print(f'Training: {tag.upper()}')
        print(f'  embed_dim={cfg.embed_dim}  depth={cfg.depth}  '
              f'd_state={cfg.d_state}  expand={cfg.expand}')
        print(f'  lr={cfg.learning_rate}  drop_path={cfg.drop_path_rate}  '
              f'layer_scale={cfg.layer_scale_init}')
        print(f'  epochs={cfg.epochs}  patience={cfg.patience}  '
              f'warmup={cfg.warmup_epochs}')
        print(f'  loss=FocalLoss(gamma=2.0)  amp=bfloat16')
        print(f'  tasks: device(w={cfg.loss_w_device}) + dist(w={cfg.loss_w_dist}) + snr(w={cfg.loss_w_snr})')
        print(f'{"="*70}')

    for epoch in range(cfg.epochs):
        lr = adjust_lr(optimizer, epoch, cfg)
        history['lr'].append(lr)

        train_loss, train_acc = train_one_epoch(
            model, train_loader,
            crit_device, crit_dist, crit_snr,
            optimizer, cfg)

        val_loss, val_metrics = evaluate(
            model, val_loader,
            crit_device, crit_dist, crit_snr, cfg)
        val_acc = val_metrics['accuracy']

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_metrics = deepcopy(val_metrics)
            patience_ctr = 0
            raw_model = model.module if isinstance(model, nn.DataParallel) \
                        else model
            best_state = deepcopy(raw_model.state_dict())
        else:
            patience_ctr += 1

        if verbose and (epoch % 5 == 0 or
                        epoch == cfg.epochs - 1 or
                        patience_ctr == 0):
            print(
                f'  Epoch {epoch+1:3d}/{cfg.epochs} | '
                f'LR {lr:.6f} | '
                f'Train Loss {train_loss:.4f} Acc {train_acc:.2f}% | '
                f'Val Loss {val_loss:.4f} Acc {val_acc:.2f}% | '
                f'Best {best_val_acc:.2f}%'
            )

        if patience_ctr >= cfg.patience:
            if verbose:
                print(f'  Early stopping at epoch {epoch+1}')
            break

    if verbose:
        print(f'\nBest Val Acc (Device ID): {best_val_acc:.2f}%')
        print(f'  Acc={best_metrics["accuracy"]:.2f}%  '
              f'F1={best_metrics["f1"]:.2f}%  '
              f'FPR={best_metrics["fpr"]:.2f}%  '
              f'FNR={best_metrics["fnr"]:.2f}%')

    del model, crit_device, crit_dist, crit_snr, optimizer
    torch.cuda.empty_cache()

    return {
        'tag':          tag,
        'cfg_snapshot': deepcopy(cfg),
        'best_val_acc': best_val_acc,
        'best_metrics': best_metrics,
        'best_state':   best_state,
        'history':      history,
    }


print('FocalLoss, training and evaluation functions defined')

In [ ]:
result = run_training(
    config,
    train_paths, train_uav, train_dist, train_snr,
    val_paths,   val_uav,   val_dist,   val_snr,
    tag='vim_genesys_final', verbose=True,
)
print(f'Best Val Acc: {result["best_val_acc"]:.2f}%')
print(f'Metrics: {result["best_metrics"]}')

os.makedirs('vim_outputs', exist_ok=True)
torch.save(result['best_state'], 'vim_outputs/vim_best.pth')
print('Weights saved to vim_outputs/vim_best.pth')

In [ ]:
@torch.no_grad()
def evaluate_at_snr(model: nn.Module,
                    paths: List[str],
                    uav_labels: np.ndarray,
                    dist_labels: np.ndarray,
                    snr_labels: np.ndarray,
                    snr_value: int,
                    cfg: Config) -> Optional[Dict]:
    mask       = snr_labels == snr_value
    snr_paths  = [paths[i] for i in range(len(paths)) if mask[i]]
    snr_uav    = uav_labels[mask]
    snr_dist   = dist_labels[mask]
    snr_snr    = snr_labels[mask]

    if len(snr_paths) == 0:
        return None

    ds     = GenesysDataset(snr_paths, snr_uav, snr_dist, snr_snr,
                            transform=get_val_transforms(cfg.img_size))
    loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=4, pin_memory=True)
    model.eval()

    all_preds, all_true = [], []
    for images, lbl_dev, _, _ in loader:
        images  = images.to(device, non_blocking=True)
        lbl_dev = lbl_dev.to(device, non_blocking=True)
        with autocast(dtype=torch.bfloat16):
            logits_dev, _, _ = model(images)
        all_preds.extend(logits_dev.argmax(1).cpu().tolist())
        all_true.extend(lbl_dev.cpu().tolist())

    return compute_metrics(np.array(all_true), np.array(all_preds), NUM_UAV)


eval_model = VisionMamba(config).to(device)
eval_model.load_state_dict(result['best_state'])
eval_model.eval()

_, eval_val_loader = build_dataloaders(
    config,
    train_paths, train_uav, train_dist, train_snr,
    val_paths,   val_uav,   val_dist,   val_snr,
)

print(f'{"="*80}')
print('SNR ROBUSTNESS EVALUATION  (held-out 20% val split, primary task: Device ID)')
print(f'{"="*80}')
print(f'{"SNR Level":>12} {"N":>6} {"Accuracy":>10} {"F1":>10} '
      f'{"FPR":>8} {"FNR":>8}')
print('-' * 65)

snr_robustness = {}

for snr_val in ALL_SNR_LEVELS:
    snr_result = evaluate_at_snr(
        eval_model, val_paths, val_uav, val_dist, val_snr, snr_val, config)

    if snr_result is None:
        print(f'  {IDX_TO_SNR[snr_val]:>10}  :  no samples in val split')
        continue

    snr_robustness[snr_val] = snr_result
    n = int((val_snr == snr_val).sum())

    print(f'  {IDX_TO_SNR[snr_val]:>10}  '
          f'  n={n:>5}  '
          f'  Acc={snr_result["accuracy"]:>5.1f}%  '
          f'  F1={snr_result["f1"]:>5.1f}%  '
          f'  FPR={snr_result["fpr"]:>5.1f}%  '
          f'  FNR={snr_result["fnr"]:>5.1f}%')

eval_crit_device = FocalLoss(
    gamma=2.0,
    label_smoothing=config.label_smoothing,
    num_classes=config.num_device_classes,
).to(device)
eval_crit_dist = FocalLoss(
    gamma=2.0,
    label_smoothing=config.label_smoothing,
    num_classes=config.num_dist_classes,
).to(device)
eval_crit_snr = FocalLoss(
    gamma=2.0,
    label_smoothing=config.label_smoothing,
    num_classes=config.num_snr_classes,
).to(device)

_, overall_metrics = evaluate(
    eval_model, eval_val_loader,
    eval_crit_device, eval_crit_dist, eval_crit_snr, config)

print(f'\n{"OVERALL (all SNRs)":>12}  '
      f'  n={len(val_paths):>5}  '
      f'  Acc={overall_metrics["accuracy"]:>5.1f}%  '
      f'  F1={overall_metrics["f1"]:>5.1f}%  '
      f'  FPR={overall_metrics["fpr"]:>5.1f}%  '
      f'  FNR={overall_metrics["fnr"]:>5.1f}%')

In [ ]:
@torch.no_grad()
def per_class_accuracy(model: nn.Module,
                        paths: List[str],
                        uav_labels: np.ndarray,
                        dist_labels: np.ndarray,
                        snr_labels: np.ndarray,
                        cfg: Config) -> Tuple[Dict[str, float], np.ndarray]:
    ds     = GenesysDataset(paths, uav_labels, dist_labels, snr_labels,
                            transform=get_val_transforms(cfg.img_size))
    loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=4, pin_memory=True)
    model.eval()
    all_preds, all_true = [], []
    for images, lbl_dev, _, _ in loader:
        images = images.to(device, non_blocking=True)
        with autocast(dtype=torch.bfloat16):
            logits_dev, _, _ = model(images)
        all_preds.extend(logits_dev.argmax(1).cpu().tolist())
        all_true.extend(lbl_dev.tolist())

    preds  = np.array(all_preds)
    truths = np.array(all_true)
    cm     = confusion_matrix(truths, preds, labels=range(NUM_UAV))
    per_class = {}
    for i, name in enumerate(UAV_NAMES):
        total   = cm[i, :].sum()
        correct = cm[i, i]
        per_class[name] = round(correct / total * 100.0, 2) if total > 0 else 0.0
    return per_class, cm


per_class, cm = per_class_accuracy(
    eval_model, val_paths, val_uav, val_dist, val_snr, config)

print(f'{"="*50}')
print('PER-CLASS ACCURACY (val split, Device ID task)')
print(f'{"="*50}')
for name, acc in per_class.items():
    bar = '#' * int(acc / 5)
    print(f'  {name:<12}: {acc:>5.1f}%  {bar}')

print(f'\nConfusion Matrix (rows=True, cols=Pred):')
header = '       ' + ''.join(f'{n[:6]:>7}' for n in UAV_NAMES)
print(header)
for i, name in enumerate(UAV_NAMES):
    row = f'{name[:6]:<7}' + ''.join(f'{cm[i,j]:>7}' for j in range(NUM_UAV))
    print(row)

In [ ]:
OUTPUT_DIR = 'vim_genesys_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

torch.save(result['best_state'],
           f'{OUTPUT_DIR}/vim_best_weights.pth')
print('Saved vim_best_weights.pth')

torch.save({
    'model_name': 'VisionMambaMultiTask',
    'cfg': {
        'embed_dim':             config.embed_dim,
        'depth':                 config.depth,
        'd_state':               config.d_state,
        'd_conv':                config.d_conv,
        'expand':                config.expand,
        'drop_rate':             config.drop_rate,
        'drop_path_rate':        config.drop_path_rate,
        'layer_scale_init':      config.layer_scale_init,
        'num_device_classes':    config.num_device_classes,
        'num_dist_classes':      config.num_dist_classes,
        'num_snr_classes':       config.num_snr_classes,
        'img_size':              config.img_size,
        'patch_size':            config.patch_size,
    },
    'uav_names':     UAV_NAMES,
    'dist_names':    DIST_NAMES,
    'snr_names':     SNR_NAMES,
    'best_val_acc':  result['best_val_acc'],
    'state_dict':    result['best_state'],
}, f'{OUTPUT_DIR}/vim_checkpoint.pth')
print('Saved vim_checkpoint.pth')

results_json = {
    'model':       'VisionMambaMultiTask',
    'dataset':     'Genesys Lab Hovering UAV Spectrogram',
    'timestamp':   datetime.now().isoformat(),
    'num_device_classes': NUM_UAV,
    'num_dist_classes':   NUM_DIST,
    'num_snr_classes':    NUM_SNR,
    'uav_names':   UAV_NAMES,
    'dist_names':  DIST_NAMES,
    'snr_names':   SNR_NAMES,
    'architecture': {
        'embed_dim':        config.embed_dim,
        'depth':            config.depth,
        'd_state':          config.d_state,
        'expand':           config.expand,
        'drop_path_rate':   config.drop_path_rate,
        'layer_scale_init': config.layer_scale_init,
    },
    'training': {
        'best_lr':        config.learning_rate,
        'batch_size':     config.batch_size,
        'epochs_ran':     len(result['history']['val_acc']),
        'warmup_epochs':  config.warmup_epochs,
        'loss_fn':        'FocalLoss(gamma=2.0)',
        'amp_dtype':      'bfloat16',
        'loss_weights':   {
            'device': config.loss_w_device,
            'dist':   config.loss_w_dist,
            'snr':    config.loss_w_snr,
        },
        'normalization': {
            'mean': SPEC_MEAN,
            'std':  SPEC_STD,
        },
    },
    'best_val_metrics':   result['best_metrics'],
    'per_class_accuracy': per_class,
    'snr_robustness': {
        IDX_TO_SNR[snr]: m for snr, m in snr_robustness.items()
    },
}

with open(f'{OUTPUT_DIR}/vim_results.json', 'w') as f:
    json.dump(results_json, f, indent=2)
print('Saved vim_results.json')

with open(f'{OUTPUT_DIR}/vim_history.pkl', 'wb') as f:
    pickle.dump(result['history'], f)
print('Saved vim_history.pkl')

zip_name = f'vim_genesys_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
shutil.make_archive(zip_name, 'zip', OUTPUT_DIR)
print(f'\nZIP: {zip_name}.zip')
print(f'Contents: {sorted(os.listdir(OUTPUT_DIR))}')

try:
    from IPython.display import FileLink, display
    display(FileLink(f'{zip_name}.zip'))
except Exception:
    pass

print(f'\n{"="*70}')
print('ALL ARTIFACTS SAVED')
print(f'  Best Val Accuracy : {result["best_val_acc"]:.2f}%')
print(f'  Best F1 Score     : {result["best_metrics"]["f1"]:.2f}%')
print(f'  Best FPR          : {result["best_metrics"]["fpr"]:.2f}%')
print(f'  Best FNR          : {result["best_metrics"]["fnr"]:.2f}%')
print(f'{"="*70}')